# EKF-Adaptive Pruning: Stage 2 - EKF Pruner + Inference

前提: Stage 1已经跑完，`./checkpoints/resnet18_cifar_base.pth` 存在 (94.97% baseline)

## 这一阶段做什么

1. 实现 `ekf_pruner.py` - 每通道独立标量EKF
2. 实现 `inference.py` - 带EKF软门控的推理pipeline
3. 跑三组对比:
   - No pruning (baseline 94.97%)
   - Random gating (随机剪同等比例)
   - EKF gating (本文方法)

剪枝范围: 只剪 layer3 和 layer4 (保守启动, 避免底层特征崩塌)。

Uncertainty proxy: 默认 softmax entropy, 代码里留接口可换。

## Step 1: 挂载Drive + 进入项目目录

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir('/content/drive/MyDrive/ekf_adaptive_pruning')
print('[cwd]', os.getcwd())
print('[files]', os.listdir())

# 检查checkpoint
ckpt_path = './checkpoints/resnet18_cifar_base.pth'
assert os.path.exists(ckpt_path), f'Checkpoint not found: {ckpt_path}'
print('[checkpoint] OK')

## Step 2: 写入 ekf_pruner.py

In [ ]:
ekf_code = '''"""EKF-based channel-wise importance tracking and soft gating."""
import torch
import torch.nn as nn
import torch.nn.functional as F


def softmax_entropy(logits):
    """Per-sample softmax entropy as uncertainty proxy."""
    p = F.softmax(logits, dim=-1)
    logp = F.log_softmax(logits, dim=-1)
    return -(p * logp).sum(dim=-1)


def max_softmax_prob(logits):
    """1 - max softmax prob as uncertainty proxy (alternative)."""
    p = F.softmax(logits, dim=-1)
    return 1.0 - p.max(dim=-1).values


class ChannelEKF:
    """Per-channel independent scalar EKF on importance proxy.

    State model: theta_{k+1} = theta_k + q,  q ~ N(0, Q)
    Obs model:   z_k = theta_k + r,           r ~ N(0, R)

    Each channel maintains (theta_hat, P) as scalar.
    """

    def __init__(self, num_channels, Q=1e-3, R=1e-2, init_theta=0.0, init_P=1.0, device="cuda"):
        self.num_channels = num_channels
        self.Q = Q
        self.R = R
        self.device = device
        self.theta = torch.full((num_channels,), init_theta, device=device)
        self.P = torch.full((num_channels,), init_P, device=device)

    def reset(self):
        self.theta.fill_(0.0)
        self.P.fill_(1.0)

    def update(self, obs):
        """obs: tensor of shape [num_channels], the observation z_k."""
        P_pred = self.P + self.Q                        # P_{k|k-1}
        K = P_pred / (P_pred + self.R)                  # Kalman gain
        self.theta = self.theta + K * (obs - self.theta)  # posterior mean
        self.P = (1 - K) * P_pred                        # posterior variance


class EKFChannelGate(nn.Module):
    """Inserted after a conv layer output, applies soft gating per channel.

    Workflow per forward pass:
      1. Compute importance proxy a_i (global avg abs activation)
      2. EKF update with a_i as observation
      3. Compute threshold tau from external uncertainty signal
      4. Apply soft gate g_i = sigmoid(alpha * (theta_hat - tau))
      5. Multiply input by g_i channel-wise
    """

    def __init__(self, num_channels, Q=1e-3, R=1e-2, alpha=10.0, tau0=0.5, tau_min=0.05, gamma=2.0, device="cuda"):
        super().__init__()
        self.num_channels = num_channels
        self.alpha = alpha
        self.tau0 = tau0
        self.tau_min = tau_min
        self.gamma = gamma
        self.ekf = ChannelEKF(num_channels, Q=Q, R=R, device=device)
        # External input: uncertainty scalar in [0, 1] (higher = more uncertain)
        self.current_uncertainty = 0.0
        # Stats for debugging/plotting
        self.last_gate = None
        self.last_keep_ratio = None

    def set_uncertainty(self, u):
        """u: scalar in [0, 1]. 0 = very confident (aggressive prune), 1 = uncertain (conservative)."""
        self.current_uncertainty = float(u)

    def compute_tau(self):
        """Threshold drops when uncertainty is high (keep more channels)."""
        return self.tau0 * torch.exp(torch.tensor(-self.gamma * self.current_uncertainty)).item() + self.tau_min

    def forward(self, x):
        # x: [B, C, H, W]
        B, C, H, W = x.shape
        # Importance proxy: global average abs activation, averaged over batch
        with torch.no_grad():
            obs = x.abs().mean(dim=[0, 2, 3])   # shape [C]
        # EKF update
        self.ekf.update(obs)
        # Threshold
        tau = self.compute_tau()
        # Soft gate
        gate = torch.sigmoid(self.alpha * (self.ekf.theta - tau))   # [C]
        # Apply channel-wise
        gate_bcast = gate.view(1, C, 1, 1)
        out = x * gate_bcast
        # Record stats
        self.last_gate = gate.detach()
        self.last_keep_ratio = (gate > 0.5).float().mean().item()
        return out


class RandomChannelGate(nn.Module):
    """Baseline: randomly zero out a fixed fraction of channels per batch."""

    def __init__(self, num_channels, keep_ratio=0.5, device="cuda"):
        super().__init__()
        self.num_channels = num_channels
        self.keep_ratio = keep_ratio
        self.last_keep_ratio = None

    def forward(self, x):
        B, C, H, W = x.shape
        n_keep = max(1, int(self.keep_ratio * C))
        perm = torch.randperm(C, device=x.device)
        mask = torch.zeros(C, device=x.device)
        mask[perm[:n_keep]] = 1.0
        self.last_keep_ratio = mask.mean().item()
        return x * mask.view(1, C, 1, 1)
'''

with open('models/ekf_pruner.py', 'w') as f:
    f.write(ekf_code)

print('写入完成:', os.path.getsize('models/ekf_pruner.py'), '字节')

## Step 3: 写入 inference.py

推理pipeline的核心做法:
- 加载训练好的ResNet18
- 在 layer3 和 layer4 的每个BasicBlock输出之后，插入gate模块
- two-pass推理: 第一遍无gate获取uncertainty, 第二遍用uncertainty调整gate

In [ ]:
inference_code = '''"""Inference with EKF-gated channel pruning."""
import os
import sys
import argparse
import time

sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname(__file__), "..")))

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as T

from models.resnet18_cifar import resnet18_cifar
from models.ekf_pruner import EKFChannelGate, RandomChannelGate, softmax_entropy


def get_test_loader(data_dir="./data", batch_size=128, num_workers=2):
    mean = (0.4914, 0.4822, 0.4465)
    std = (0.2470, 0.2435, 0.2616)
    test_tf = T.Compose([T.ToTensor(), T.Normalize(mean, std)])
    test_set = torchvision.datasets.CIFAR10(root=data_dir, train=False, download=True, transform=test_tf)
    return DataLoader(test_set, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)


def wrap_layer_with_gate(layer_seq, gate_cls, gate_kwargs, device):
    """Insert a gate after each BasicBlock in a Sequential.

    layer_seq: nn.Sequential of BasicBlocks (e.g. model.layer3)
    Returns new nn.Sequential with gates interleaved, plus a list of gate modules.
    """
    new_modules = []
    gate_list = []
    for block in layer_seq:
        new_modules.append(block)
        # Infer output channels from block.conv2
        out_ch = block.conv2.out_channels
        gate = gate_cls(num_channels=out_ch, device=device, **gate_kwargs).to(device)
        new_modules.append(gate)
        gate_list.append(gate)
    return nn.Sequential(*new_modules), gate_list


def install_gates(model, gate_cls, gate_kwargs, target_layers=("layer3", "layer4"), device="cuda"):
    """Install gates on selected layers of the model, return list of gates."""
    all_gates = []
    for lname in target_layers:
        layer = getattr(model, lname)
        new_seq, gates = wrap_layer_with_gate(layer, gate_cls, gate_kwargs, device)
        setattr(model, lname, new_seq)
        all_gates.extend(gates)
    return all_gates


def evaluate_nogate(model, loader, device):
    """Plain evaluation (no gating)."""
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            pred = model(x).argmax(dim=1)
            correct += (pred == y).sum().item()
            total += y.size(0)
    return 100.0 * correct / total


def evaluate_with_ekf_gates(model, gates, loader, device, entropy_norm=2.3):
    """Two-pass inference:
      Pass 1: disable gates (set uncertainty=1 to keep everything), get entropy.
      Pass 2: use entropy as uncertainty signal, re-infer with gating.

    For simplicity here we approximate via single-pass using running uncertainty
    averaged over the previous batch. First batch just uses uncertainty=0.5.
    """
    model.eval()
    correct, total = 0, 0
    keep_ratios = [[] for _ in gates]
    running_unc = 0.5  # warmup
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            # Set uncertainty on gates based on previous batch
            for g in gates:
                g.set_uncertainty(running_unc)
            logits = model(x)
            pred = logits.argmax(dim=1)
            correct += (pred == y).sum().item()
            total += y.size(0)
            # Compute uncertainty for next batch: mean entropy, normalized to [0,1]
            ent = softmax_entropy(logits).mean().item()
            running_unc = min(1.0, ent / entropy_norm)
            for i, g in enumerate(gates):
                if g.last_keep_ratio is not None:
                    keep_ratios[i].append(g.last_keep_ratio)
    acc = 100.0 * correct / total
    avg_keep = [sum(r) / len(r) if r else 0.0 for r in keep_ratios]
    return acc, avg_keep


def evaluate_with_random_gates(model, gates, loader, device):
    model.eval()
    correct, total = 0, 0
    keep_ratios = [[] for _ in gates]
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            pred = model(x).argmax(dim=1)
            correct += (pred == y).sum().item()
            total += y.size(0)
            for i, g in enumerate(gates):
                if g.last_keep_ratio is not None:
                    keep_ratios[i].append(g.last_keep_ratio)
    acc = 100.0 * correct / total
    avg_keep = [sum(r) / len(r) if r else 0.0 for r in keep_ratios]
    return acc, avg_keep


def load_base_model(ckpt_path, device):
    model = resnet18_cifar(num_classes=10).to(device)
    ckpt = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])
    return model


def run_comparison(ckpt_path, device="cuda"):
    loader = get_test_loader()

    # Baseline (no pruning)
    print("\\n=== Baseline (no pruning) ===")
    m = load_base_model(ckpt_path, device)
    acc = evaluate_nogate(m, loader, device)
    print(f"acc = {acc:.2f}%")

    # EKF gating
    print("\\n=== EKF adaptive gating ===")
    m = load_base_model(ckpt_path, device)
    ekf_kwargs = dict(Q=1e-3, R=1e-2, alpha=10.0, tau0=0.3, tau_min=0.05, gamma=2.0)
    gates = install_gates(m, EKFChannelGate, ekf_kwargs, target_layers=("layer3", "layer4"), device=device)
    acc_ekf, keep_ekf = evaluate_with_ekf_gates(m, gates, loader, device)
    avg_keep_ekf = sum(keep_ekf) / len(keep_ekf)
    print(f"acc = {acc_ekf:.2f}%,  avg_keep_ratio = {avg_keep_ekf:.3f}")
    print(f"per-gate keep ratios: {[f\'{r:.3f}\' for r in keep_ekf]}")

    # Random gating at same average keep ratio as EKF
    print("\\n=== Random gating (matched keep ratio) ===")
    m = load_base_model(ckpt_path, device)
    rand_kwargs = dict(keep_ratio=avg_keep_ekf)
    gates = install_gates(m, RandomChannelGate, rand_kwargs, target_layers=("layer3", "layer4"), device=device)
    acc_rand, keep_rand = evaluate_with_random_gates(m, gates, loader, device)
    print(f"acc = {acc_rand:.2f}%,  avg_keep_ratio = {sum(keep_rand)/len(keep_rand):.3f}")

    print("\\n=== Summary ===")
    print(f"  No pruning:      {acc:.2f}%")
    print(f"  EKF gating:      {acc_ekf:.2f}%  (keep {avg_keep_ekf:.2%})")
    print(f"  Random (match):  {acc_rand:.2f}%  (keep {avg_keep_ekf:.2%})")


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--ckpt", type=str, default="./checkpoints/resnet18_cifar_base.pth")
    args = parser.parse_args()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"[device] {device}")
    run_comparison(args.ckpt, device=device)
'''

with open('src/inference.py', 'w') as f:
    f.write(inference_code)

print('写入完成:', os.path.getsize('src/inference.py'), '字节')

## Step 4: Smoke test EKF模块

In [ ]:
import sys
sys.path.insert(0, '.')

# Clear import cache
for mod in list(sys.modules.keys()):
    if mod.startswith('models') or mod.startswith('src'):
        del sys.modules[mod]

import torch
from models.ekf_pruner import ChannelEKF, EKFChannelGate, softmax_entropy

# Test ChannelEKF
ekf = ChannelEKF(num_channels=16, Q=1e-3, R=1e-2)
for i in range(10):
    obs = torch.rand(16, device='cuda')
    ekf.update(obs)
print(f'EKF theta: mean={ekf.theta.mean().item():.4f}, std={ekf.theta.std().item():.4f}')
print(f'EKF P:     mean={ekf.P.mean().item():.4f}  (should decrease over updates)')

# Test EKFChannelGate
gate = EKFChannelGate(num_channels=16).cuda()
gate.set_uncertainty(0.5)
x = torch.randn(4, 16, 8, 8, device='cuda')
y = gate(x)
print(f'Input shape:  {x.shape}')
print(f'Output shape: {y.shape}')
print(f'Last gate (5 ch): {gate.last_gate[:5].tolist()}')
print(f'Keep ratio:       {gate.last_keep_ratio:.3f}')

# Test entropy
logits = torch.randn(8, 10, device='cuda')
ent = softmax_entropy(logits)
print(f'Entropy (8 samples): mean={ent.mean().item():.4f}')

## Step 5: 跑对比实验

三组对比: No pruning / EKF gating / Random gating。

预计时间: 3次推理, 每次约1分钟, 总共3-5分钟。

In [ ]:
!python src/inference.py

## 预期结果

- **No pruning**: 94.97% (和training时的best_acc一致)
- **EKF gating**: 某个accuracy + keep_ratio
- **Random gating**: 通常比EKF低很多 (因为随机会把重要channel也剪掉)

**第一版不强求EKF完美**, 只要:
1. EKF能跑通 (没报错)
2. EKF > Random (说明机制work)
3. keep_ratio在合理范围 (50%-90%)

之后可以调 `tau0`, `alpha`, `Q`, `R` 来做trade-off曲线。